# 03b — ProtoNet (Metric Learning)

**Paradigm:** Metric learning via prototypical networks (Snell et al., 2017)  
**Encoder:** CNN backbone loaded from `models/custom_cnn/best_model.pt` (frozen first, then fine-tuned)  
**Embedding:** 256-dim L2-normalised projection head  
**Episodes:** N=7-way, K=5-shot, Q=15 queries/class  
**Loss:** Prototypical cross-entropy + SupCon auxiliary (λ=0.1)  
**Curriculum:** Same three-phase SNR schedule as CNN (A→B→C)  
**Target:** macro F1 > 0.80, AUC-ROC > 0.85 per class

In [19]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import json, random, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler, ConcatDataset
from tqdm import tqdm
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    roc_auc_score, average_precision_score
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cuda


In [20]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
CFG = {
    # paths
    'data_root':       Path('/kaggle/input/datasets/orpheusmanga/alertreck-mel2/mel'),
    'cnn_checkpoint':  Path('/kaggle/input/datasets/orpheusmanga/alertreck-cnn-checkpoint/models/custom_cnn/best_model.pt'),
    'output_dir':      Path('/kaggle/working/protonet'),
    # data
    'n_classes':       7,
    'n_mels':          128,
    'n_frames':        301,
    'label_names':     ['background_animals','background_wind_rain','threat_chainsaw',
                        'threat_dog','threat_gunshot','threat_human','threat_vehicle'],
    'seed':            42,
    'batch_size':      256,
    # episode design
    'n_way':           7,
    'k_shot':          5,
    'n_query':         15,
    'episodes_per_epoch': 200,
    # embedding
    'embed_dim':       256,
    # training
    'freeze_epochs':   5,
    'epochs':          50,
    'lr':              1e-4,
    'lr_head':         1e-3,
    'weight_decay':    1e-4,
    'patience':        12,
    'supcon_lambda':   0.1,
    'supcon_temp':     0.07,
    'phase_epochs':    {'A': 20, 'B': 15, 'C': 999},
}

CFG['output_dir'].mkdir(parents=True, exist_ok=True)

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
random.seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

print('CFG loaded.')
print(f'data_root : {CFG["data_root"]}')
print(f'checkpoint: {CFG["cnn_checkpoint"]}')


CFG loaded.
data_root : /kaggle/input/datasets/orpheusmanga/alertreck-mel2/mel
checkpoint: /kaggle/input/datasets/orpheusmanga/alertreck-cnn-checkpoint/models/custom_cnn/best_model.pt


In [21]:
# ── 3. Dataset ────────────────────────────────────────────────────────────────
class ShardDataset(Dataset):
    """
    Loads all .npz shards from a split directory into memory.
    Each shard contains X (N, 128, n_frames) float32 and y (N,) int64.
    """
    def __init__(self, shard_dir: Path, augment: bool = False):
        shards = sorted(shard_dir.glob('*.npz'))
        assert shards, f'No shards found in {shard_dir}'

        xs, ys = [], []
        for s in tqdm(shards, desc=f'Loading {shard_dir.name}'):
            d = np.load(s)
            xs.append(d['X'])
            ys.append(d['y'].astype(np.int64))

        self.X      = np.concatenate(xs, axis=0)
        self.y      = np.concatenate(ys, axis=0)
        self.augment = augment
        print(f'  {shard_dir.name}: {len(self.X):,} samples')

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].copy()
        mean_val = x.mean()

        if self.augment:
            for _ in range(2):
                f_width = random.randint(0, 20)
                f0      = random.randint(0, x.shape[0] - f_width)
                x[f0:f0 + f_width, :] = mean_val
            for _ in range(2):
                t_width = random.randint(0, 40)
                t0      = random.randint(0, x.shape[1] - t_width)
                x[:, t0:t0 + t_width] = mean_val

        x = torch.from_numpy(x).unsqueeze(0)          # (1, 128, n_frames)
        y = torch.tensor(self.y[idx], dtype=torch.long)
        return x, y


data_root = CFG['data_root']

print('Loading val / test …')
val_ds  = ShardDataset(data_root / 'val',  augment=False)
test_ds = ShardDataset(data_root / 'test', augment=False)

print('\nLoading curriculum phases …')
train_clean_ds  = ShardDataset(data_root / 'train',       augment=True)
train_aug_A_ds  = ShardDataset(data_root / 'train_aug_A', augment=True)
train_aug_B_ds  = ShardDataset(data_root / 'train_aug_B', augment=True)
train_aug_C_ds  = ShardDataset(data_root / 'train_aug_C', augment=True)

# Phase datasets: clean + augmented (same as CNN notebook)
phase_datasets = {
    'A': ConcatDataset([train_clean_ds, train_aug_A_ds]),
    'B': ConcatDataset([train_clean_ds, train_aug_B_ds]),
    'C': ConcatDataset([train_clean_ds, train_aug_C_ds]),
}

val_loader  = DataLoader(val_ds,  batch_size=CFG['batch_size'], shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=4, pin_memory=True)

print(f'\nVal  : {len(val_ds):,} samples')
print(f'Test : {len(test_ds):,} samples')
print(f'Phase A train : {len(phase_datasets["A"]):,} samples')
print(f'Phase B train : {len(phase_datasets["B"]):,} samples')
print(f'Phase C train : {len(phase_datasets["C"]):,} samples')


Loading val / test …


Loading val: 100%|██████████| 4/4 [00:04<00:00,  1.17s/it]


  val: 3,392 samples


Loading test: 100%|██████████| 4/4 [00:04<00:00,  1.08s/it]


  test: 3,439 samples

Loading curriculum phases …


...

  train_aug_A: 10,223 samples


Loading train_aug_B: 100%|██████████| 21/21 [00:21<00:00,  1.04s/it]


  train_aug_B: 20,446 samples


Loading train_aug_C: 100%|██████████| 31/31 [00:26<00:00,  1.19it/s]


  train_aug_C: 30,669 samples

Val  : 3,392 samples
Test : 3,439 samples
Phase A train : 20,446 samples
Phase B train : 30,669 samples
Phase C train : 40,892 samples


In [23]:
# ── 4. Model: CNN Encoder + Projection Head ───────────────────────────────────

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout),
        )
    def forward(self, x): return self.block(x)


class AudioCNN(nn.Module):
    """Exact replica of CNN notebook architecture — needed to load checkpoint."""
    def __init__(self, n_classes: int = 7):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(1,   32,  dropout=0.2),
            ConvBlock(32,  64,  dropout=0.2),
            ConvBlock(64,  128, dropout=0.2),
            ConvBlock(128, 256, dropout=0.2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.pool(self.encoder(x)).flatten(1))
    def encode(self, x):
        return self.pool(self.encoder(x)).flatten(1)


class ProtoNet(nn.Module):
    """
    CNN encoder (pretrained) + lightweight projection head.
    embed_dim: 256 → 256 with L2 normalisation.
    """
    def __init__(self, cnn_checkpoint: Path, embed_dim: int = 256, n_classes: int = 7):
        super().__init__()
        cnn = AudioCNN(n_classes=n_classes)
        ckpt = torch.load(cnn_checkpoint, map_location='cpu', weights_only=True)
        # checkpoint may be a full training snapshot or a raw state_dict
        state = ckpt.get('model_state', ckpt)
        cnn.load_state_dict(state)
        self.encoder = cnn.encoder
        self.pool    = cnn.pool

        self.proj = nn.Sequential(
            nn.Linear(256, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, embed_dim, bias=False),
        )

        self.register_buffer('prototypes', torch.zeros(n_classes, embed_dim))
        self.n_classes = n_classes
        self.embed_dim = embed_dim

    def embed(self, x):
        feat = self.pool(self.encoder(x)).flatten(1)
        return F.normalize(self.proj(feat), dim=1)

    def forward(self, x):
        return self.embed(x) @ self.prototypes.T

    def freeze_encoder(self):
        for p in self.encoder.parameters(): p.requires_grad_(False)
        for p in self.pool.parameters():    p.requires_grad_(False)

    def unfreeze_encoder(self):
        for p in self.encoder.parameters(): p.requires_grad_(True)
        for p in self.pool.parameters():    p.requires_grad_(True)


model = ProtoNet(
    cnn_checkpoint=CFG['cnn_checkpoint'],
    embed_dim=CFG['embed_dim'],
    n_classes=CFG['n_classes'],
).to(device)

model.freeze_encoder()

n_params    = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {n_params:,}')
print(f'Trainable params: {n_trainable:,}  (encoder frozen)')

dummy = torch.randn(4, 1, CFG['n_mels'], CFG['n_frames']).to(device)
emb   = model.embed(dummy)
print(f'Embed shape     : {tuple(emb.shape)}')
print(f'L2 norms        : {emb.norm(dim=1).tolist()}')


Total params    : 1,304,224
Trainable params: 131,584  (encoder frozen)
Embed shape     : (4, 256)
L2 norms        : [0.9999999403953552, 1.0, 1.0, 0.9999999403953552]


In [28]:
# ── 5. Loss Functions ─────────────────────────────────────────────────────────

def prototypical_loss(support_emb, support_labels, query_emb, query_labels, n_way, k_shot):
    """
    Prototypical cross-entropy loss.
    support_emb : (n_way*k_shot, D) — L2-normalised
    query_emb   : (n_way*n_query, D)
    """
    protos = torch.stack([
        support_emb[support_labels == c].mean(0)
        for c in range(n_way)
    ])
    protos = F.normalize(protos, dim=1)

    logits = 10.0 * (query_emb @ protos.T)  # (n_query_total, n_way)

    unique  = torch.unique(support_labels, sorted=True)
    remap   = {c.item(): i for i, c in enumerate(unique)}
    local_q = torch.tensor([remap[l.item()] for l in query_labels],
                            dtype=torch.long, device=query_emb.device)

    loss = F.cross_entropy(logits, local_q)
    acc  = (logits.argmax(1) == local_q).float().mean().item()
    return loss, acc, logits, protos


def supcon_loss(embeddings, labels, temperature: float = 0.07):
    """
    Supervised contrastive loss (Khosla et al., 2020).
    embeddings : (N, D) — L2-normalised
    labels     : (N,)
    """
    N   = embeddings.size(0)
    sim = (embeddings @ embeddings.T) / temperature  # (N, N)

    labels   = labels.view(-1, 1)
    pos_mask = (labels == labels.T).float()
    pos_mask.fill_diagonal_(0)

    # numerically stable: subtract row max
    sim_max = sim.detach().max(dim=1, keepdim=True).values
    sim     = sim - sim_max

    exp_sim = torch.exp(sim)
    # zero out self-similarity WITHOUT in-place op (avoids autograd corruption)
    self_mask = ~torch.eye(N, dtype=torch.bool, device=embeddings.device)
    exp_sim   = exp_sim * self_mask.float()

    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)

    n_pos = pos_mask.sum(dim=1).clamp(min=1)
    loss  = -(pos_mask * log_prob).sum(dim=1) / n_pos
    return loss.mean()


print('Loss functions defined.')


Loss functions defined.


In [29]:
# ── 6. Episodic Sampler ───────────────────────────────────────────────────────

def build_class_index(dataset):
    """
    Build class → [indices] map.
    Works with ShardDataset (.y numpy array) and ConcatDataset of ShardDatasets.
    """
    idx_by_class = defaultdict(list)
    if hasattr(dataset, 'y'):
        # ShardDataset
        for i, label in enumerate(dataset.y):
            idx_by_class[int(label)].append(i)
    else:
        # ConcatDataset — walk each constituent dataset
        offset = 0
        for ds in dataset.datasets:
            for i, label in enumerate(ds.y):
                idx_by_class[int(label)].append(offset + i)
            offset += len(ds)
    return idx_by_class


class EpisodicBatchSampler(Sampler):
    """
    Yields one flat index list per episode:
    [class0_s0..class0_sK, class0_q0..class0_qQ, class1_s0..., ...]
    """
    def __init__(self, idx_by_class, n_way, k_shot, n_query, n_episodes):
        self.idx_by_class = idx_by_class
        self.n_way      = n_way
        self.k_shot     = k_shot
        self.n_query    = n_query
        self.n_episodes = n_episodes

    def __len__(self):
        return self.n_episodes

    def __iter__(self):
        for _ in range(self.n_episodes):
            classes = random.sample(list(self.idx_by_class.keys()), self.n_way)
            episode = []
            for c in classes:
                chosen = random.sample(self.idx_by_class[c], self.k_shot + self.n_query)
                episode.extend(chosen)
            yield episode


def make_episode_loader(dataset):
    idx = build_class_index(dataset)
    sampler = EpisodicBatchSampler(
        idx_by_class=idx,
        n_way=CFG['n_way'],
        k_shot=CFG['k_shot'],
        n_query=CFG['n_query'],
        n_episodes=CFG['episodes_per_epoch'],
    )
    return DataLoader(dataset, batch_sampler=sampler, num_workers=4, pin_memory=True)


def parse_episode(batch):
    """Split flat episodic batch into support / query tensors."""
    X, y   = batch
    N, K, Q = CFG['n_way'], CFG['k_shot'], CFG['n_query']
    X = X.view(N, K + Q, *X.shape[1:])
    y = y.view(N, K + Q)
    support_x = X[:, :K].reshape(N*K, *X.shape[2:]).to(device)
    query_x   = X[:, K:].reshape(N*Q, *X.shape[2:]).to(device)
    support_y = y[:, :K].reshape(N*K).to(device)
    query_y   = y[:, K:].reshape(N*Q).to(device)
    return support_x, support_y, query_x, query_y


print('Episodic sampler defined.')


Episodic sampler defined.


In [30]:
# ── 7. Optimiser & Scheduler ──────────────────────────────────────────────────

encoder_params    = list(model.encoder.parameters()) + list(model.pool.parameters())
projection_params = list(model.proj.parameters())

optimizer = optim.AdamW([
    {'params': projection_params, 'lr': CFG['lr_head']},
    {'params': encoder_params,    'lr': CFG['lr']},
], weight_decay=CFG['weight_decay'])

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=1e-6,
)

print('Optimiser: AdamW | head lr={:.0e} | encoder lr={:.0e}'.format(
      CFG['lr_head'], CFG['lr']))
print(f'Scheduler: CosineAnnealingLR T_max={CFG["epochs"]}')

Optimiser: AdamW | head lr=1e-03 | encoder lr=1e-04
Scheduler: CosineAnnealingLR T_max=50


In [31]:
# ── 8. Training Loop ──────────────────────────────────────────────────────────

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_f1':    [],
    'lr': [], 'phase': [],
}

best_val_f1  = 0.0
patience_cnt = 0
best_epoch   = 0
best_phase   = 'A'


def get_phase(epoch):
    if epoch < CFG['phase_epochs']['A']:                              return 'A'
    if epoch < CFG['phase_epochs']['A'] + CFG['phase_epochs']['B']:  return 'B'
    return 'C'


def evaluate(loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits     = model(X)
            loss       = F.cross_entropy(logits, y)
            total_loss += loss.item() * len(y)
            n          += len(y)
            all_probs.append(F.softmax(logits, dim=1).cpu())
            all_preds.append(logits.argmax(1).cpu())
            all_labels.append(y.cpu())
    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    probs  = torch.cat(all_probs).numpy()
    f1     = f1_score(labels, preds, average='macro', zero_division=0)
    return total_loss / n, preds, labels, probs, f1


def update_prototypes(dataset):
    """Recompute class prototypes over a full dataset and store in model buffer."""
    model.eval()
    loader = DataLoader(dataset, batch_size=512, shuffle=False,
                        num_workers=4, pin_memory=True)
    sums   = torch.zeros(CFG['n_classes'], CFG['embed_dim'], device=device)
    counts = torch.zeros(CFG['n_classes'], device=device)
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            z = model.embed(X)
            for c in range(CFG['n_classes']):
                mask = (y == c)
                if mask.any():
                    sums[c]   += z[mask].sum(0)
                    counts[c] += mask.sum()
    protos = F.normalize(sums / counts.unsqueeze(1).clamp(min=1), dim=1)
    model.prototypes.copy_(protos)


# ── initialise with Phase A ───────────────────────────────────────────────────
current_phase = 'A'
ep_loader     = make_episode_loader(phase_datasets['A'])

print(f'Starting training — {CFG["epochs"]} epochs')
print(f'Phase A: {CFG["phase_epochs"]["A"]} epochs  |  '
      f'Phase B: {CFG["phase_epochs"]["B"]} epochs  |  Phase C: remaining')
print(f'Encoder frozen for first {CFG["freeze_epochs"]} epochs\n')

for epoch in range(CFG['epochs']):
    t0 = time.time()

    # curriculum phase switch
    phase = get_phase(epoch)
    if phase != current_phase:
        current_phase = phase
        ep_loader     = make_episode_loader(phase_datasets[phase])
        print(f'  >> Switched to Phase {phase} at epoch {epoch}')

    # unfreeze encoder
    if epoch == CFG['freeze_epochs']:
        model.unfreeze_encoder()
        n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  >> Encoder unfrozen at epoch {epoch} | trainable: {n_tr:,}')

    # episodic training
    model.train()
    ep_losses, ep_accs = [], []
    for batch in ep_loader:
        sup_x, sup_y, qry_x, qry_y = parse_episode(batch)

        sup_z = model.embed(sup_x)
        qry_z = model.embed(qry_x)

        proto_loss, ep_acc, _, _ = prototypical_loss(
            sup_z, sup_y, qry_z, qry_y, CFG['n_way'], CFG['k_shot'],
        )
        sc_loss = supcon_loss(torch.cat([sup_z, qry_z]),
                              torch.cat([sup_y, qry_y]), CFG['supcon_temp'])
        loss = proto_loss + CFG['supcon_lambda'] * sc_loss

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        ep_losses.append(loss.item())
        ep_accs.append(ep_acc)

    scheduler.step()

    tr_loss = np.mean(ep_losses)
    tr_acc  = np.mean(ep_accs)

    # update prototypes then validate
    update_prototypes(phase_datasets[phase])
    va_loss, _, _, _, va_f1 = evaluate(val_loader)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_f1'].append(va_f1)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    history['phase'].append(phase)

    elapsed = time.time() - t0
    print(f'Ep {epoch+1:03d}/{CFG["epochs"]} [{phase}] '
          f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} '
          f'va_loss={va_loss:.4f} va_f1={va_f1:.4f} '
          f'lr={optimizer.param_groups[0]["lr"]:.2e} ({elapsed:.1f}s)')

    if va_f1 > best_val_f1:
        best_val_f1  = va_f1
        best_epoch   = epoch + 1
        best_phase   = phase
        patience_cnt = 0
        torch.save(model.state_dict(), CFG['output_dir'] / 'best_model.pt')
        print(f'  >> Saved best (val_f1={best_val_f1:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= CFG['patience']:
            print(f'  >> Early stop at epoch {epoch+1}; best was epoch {best_epoch}')
            break

print(f'\nTraining complete. Best val_f1={best_val_f1:.4f} at epoch {best_epoch} (Phase {best_phase})')


Starting training — 50 epochs
Phase A: 20 epochs  |  Phase B: 15 epochs  |  Phase C: remaining
Encoder frozen for first 5 epochs

Ep 001/50 [A] tr_loss=0.7066 tr_acc=0.8950 va_loss=1.4035 va_f1=0.9162 lr=9.99e-04 (69.9s)
  >> Saved best (val_f1=0.9162)
Ep 002/50 [A] tr_loss=0.6672 tr_acc=0.9040 va_loss=1.4333 va_f1=0.9137 lr=9.96e-04 (73.3s)
Ep 003/50 [A] tr_loss=0.6471 tr_acc=0.9090 va_loss=1.4197 va_f1=0.9112 lr=9.91e-04 (72.7s)
Ep 004/50 [A] tr_loss=0.6503 tr_acc=0.9073 va_loss=1.4229 va_f1=0.9209 lr=9.84e-04 (72.8s)
  >> Saved best (val_f1=0.9209)
Ep 005/50 [A] tr_loss=0.6412 tr_acc=0.9084 va_loss=1.4422 va_f1=0.9215 lr=9.76e-04 (72.8s)
  >> Saved best (val_f1=0.9215)
  >> Encoder unfrozen at epoch 5 | trainable: 1,304,224
Ep 006/50 [A] tr_loss=0.6087 tr_acc=0.9171 va_loss=1.4226 va_f1=0.9248 lr=9.65e-04 (157.3s)
  >> Saved best (val_f1=0.9248)
Ep 007/50 [A] tr_loss=0.5972 tr_acc=0.9201 va_loss=1.4149 va_f1=0.9297 lr=9.52e-04 (157.2s)
  >> Saved best (val_f1=0.9297)
Ep 008/50 [A] t

In [33]:
# ── 10. Test Evaluation ───────────────────────────────────────────────────────
state = torch.load(CFG['output_dir'] / 'best_model.pt', map_location=device, weights_only=True)
model.load_state_dict(state)

# Recompute prototypes on full Phase-C training set
update_prototypes(phase_datasets['C'])
print('Prototypes recomputed on Phase-C training set.')

te_loss, te_preds, te_labels, te_probs, te_f1 = evaluate(test_loader)
te_acc = (te_preds == te_labels).mean()

print(f'\nTest accuracy   : {te_acc:.4f}')
print(f'Test macro F1   : {te_f1:.4f}')
print(f'\n{classification_report(te_labels, te_preds, target_names=CFG["label_names"], digits=4)}')


Prototypes recomputed on Phase-C training set.

Test accuracy   : 0.9334
Test macro F1   : 0.9244

                      precision    recall  f1-score   support

  background_animals     0.9617    0.8874    0.9231      1190
background_wind_rain     0.9170    0.9338    0.9253       272
     threat_chainsaw     0.9589    0.9103    0.9340       513
          threat_dog     0.7481    0.9019    0.8178       214
      threat_gunshot     0.9917    1.0000    0.9959       480
        threat_human     0.9244    0.9982    0.9599       551
      threat_vehicle     0.8750    0.9589    0.9150       219

            accuracy                         0.9334      3439
           macro avg     0.9110    0.9415    0.9244      3439
        weighted avg     0.9372    0.9334    0.9339      3439



In [34]:
# ── 11. Confusion Matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(te_labels, te_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(CFG['n_classes'])); ax.set_yticks(range(CFG['n_classes']))
ax.set_xticklabels(CFG['label_names'], rotation=45, ha='right')
ax.set_yticklabels(CFG['label_names'])
ax.set(xlabel='Predicted', ylabel='True', title='Confusion Matrix (row-normalised) — ProtoNet')
for i in range(CFG['n_classes']):
    for j in range(CFG['n_classes']):
        ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                color='white' if cm_norm[i,j] > 0.6 else 'black', fontsize=8)
plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'confusion_matrix.png', dpi=150)
print('Saved confusion_matrix.png')

Saved confusion_matrix.png


In [35]:
# ── 12. AUC-ROC + Precision/Recall Evaluation ─────────────────────────────────
from sklearn.metrics import roc_curve, precision_recall_curve

classes = list(range(CFG['n_classes']))
y_bin   = label_binarize(te_labels, classes=classes)  # (N, 7)

per_class_auc = {}
per_class_ap  = {}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ROC curves
for c, name in enumerate(CFG['label_names']):
    fpr, tpr, _ = roc_curve(y_bin[:, c], te_probs[:, c])
    auc = roc_auc_score(y_bin[:, c], te_probs[:, c])
    per_class_auc[name] = auc
    axes[0].plot(fpr, tpr, label=f'{name} ({auc:.3f})')
axes[0].axvline(0.20, ls='--', color='grey', alpha=0.5, label='FPR=20% limit')
axes[0].plot([0,1],[0,1], 'k--', alpha=0.3)
axes[0].set(xlabel='FPR', ylabel='TPR', title='ROC Curves (one-vs-rest) — ProtoNet')
axes[0].legend(fontsize=8)

# PR curves
for c, name in enumerate(CFG['label_names']):
    prec, rec, _ = precision_recall_curve(y_bin[:, c], te_probs[:, c])
    ap = average_precision_score(y_bin[:, c], te_probs[:, c])
    per_class_ap[name] = ap
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.3f})')
axes[1].set(xlabel='Recall', ylabel='Precision', title='PR Curves — ProtoNet')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'roc_pr_curves.png', dpi=150)
print('Saved roc_pr_curves.png')

macro_auc = np.mean(list(per_class_auc.values()))
macro_ap  = np.mean(list(per_class_ap.values()))
print(f'\nMacro AUC-ROC : {macro_auc:.4f}  (target >0.85)')
print(f'Macro AP      : {macro_ap:.4f}')
print('\nPer-class AUC-ROC:')
for name, auc in sorted(per_class_auc.items(), key=lambda x: x[1]):
    flag = '  OK' if auc >= 0.85 else '  BELOW TARGET'
    print(f'  {name:<30} {auc:.4f}{flag}')

Saved roc_pr_curves.png

Macro AUC-ROC : 0.9928  (target >0.85)
Macro AP      : 0.9754

Per-class AUC-ROC:
  threat_dog                     0.9827  OK
  background_animals             0.9883  OK
  threat_chainsaw                0.9898  OK
  background_wind_rain           0.9904  OK
  threat_vehicle                 0.9986  OK
  threat_human                   0.9995  OK
  threat_gunshot                 1.0000  OK


In [36]:
# ── 13. ONNX Export ───────────────────────────────────────────────────────────
onnx_path = CFG['output_dir'] / 'protonet.onnx'

model.eval()
dummy_input = torch.randn(1, 1, CFG['n_mels'], CFG['n_frames']).to(device)

torch.onnx.export(
    model, dummy_input, onnx_path,
    dynamo=False,         # legacy TorchScript exporter — no onnxscript dep
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['mel_spectrogram'],
    output_names=['class_logits'],
    dynamic_axes={
        'mel_spectrogram': {0: 'batch_size'},
        'class_logits':    {0: 'batch_size'},
    },
)
print(f'ONNX model exported → {onnx_path}')
print(f'File size: {onnx_path.stat().st_size / 1e6:.2f} MB')

/tmp/ipykernel_58/1674059861.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX model exported → /kaggle/working/protonet/protonet.onnx
File size: 5.23 MB


In [37]:
# ── 14. Save Results ──────────────────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score

per_class_f1   = f1_score(te_labels, te_preds, average=None, zero_division=0)
per_class_prec = precision_score(te_labels, te_preds, average=None, zero_division=0)
per_class_rec  = recall_score(te_labels, te_preds, average=None, zero_division=0)

results = {
    # metadata
    'model':         'ProtoNet',
    'best_epoch':    best_epoch,
    'best_phase':    best_phase,
    'best_val_f1':   best_val_f1,
    'n_params':      sum(p.numel() for p in model.parameters()),
    # CFG snapshot
    'n_way':         CFG['n_way'],
    'k_shot':        CFG['k_shot'],
    'n_query':       CFG['n_query'],
    'embed_dim':     CFG['embed_dim'],
    'supcon_lambda': CFG['supcon_lambda'],
    # test metrics
    'test_acc':      float(te_acc),
    'test_macro_f1': float(te_f1),
    'macro_auc':     float(macro_auc),
    'macro_ap':      float(macro_ap),
    # per-class
    'per_class_f1':  {n: float(v) for n, v in zip(CFG['label_names'], per_class_f1)},
    'per_class_prec':{n: float(v) for n, v in zip(CFG['label_names'], per_class_prec)},
    'per_class_rec': {n: float(v) for n, v in zip(CFG['label_names'], per_class_rec)},
    'per_class_auc': per_class_auc,
    'per_class_ap':  per_class_ap,
    # training history
    'history':       history,
}

with open(CFG['output_dir'] / 'results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Also save model config for cross-notebook reference
cfg_save = {
    'n_classes': CFG['n_classes'],
    'n_mels':    CFG['n_mels'],
    'n_frames':  CFG['n_frames'],
    'embed_dim': CFG['embed_dim'],
    'label_names': CFG['label_names'],
    'test_acc':   float(te_acc),
    'test_macro_f1': float(te_f1),
    'macro_auc':  float(macro_auc),
    'best_epoch': best_epoch,
    'best_val_f1': best_val_f1,
    'best_phase': best_phase,
}
with open(CFG['output_dir'] / 'model_config_protonet.json', 'w') as f:
    json.dump(cfg_save, f, indent=2)

print('Results saved to models/protonet/')
print(f'  test_acc      : {te_acc:.4f}')
print(f'  test_macro_f1 : {te_f1:.4f}')
print(f'  macro_auc     : {macro_auc:.4f}')
print(f'  best_val_f1   : {best_val_f1:.4f}  (epoch {best_epoch}, Phase {best_phase})')

Results saved to models/protonet/
  test_acc      : 0.9334
  test_macro_f1 : 0.9244
  macro_auc     : 0.9928
  best_val_f1   : 0.9400  (epoch 31, Phase B)
